# 04 — Collect results into a Markdown report

Reads every `results/*.json`, prints the summary table, saves `results/summary.md` and the comparison plots.
Paste the table into the README's "Results" section.

In [ ]:
# @title Setup — clone repo (if needed), install deps, detect GPU
import os, sys, subprocess, json, time, math
REPO_URL = "https://github.com/YOUR_GITHUB_USER/reversible-llm-poc.git"   # <-- edit after you push

if not os.path.exists("src/revllm.py"):
    if os.path.exists("../src/revllm.py"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "reversible-llm-poc"], check=True)
        os.chdir("reversible-llm-poc")
sys.path.insert(0, os.path.abspath("src"))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tiktoken", "datasets", "matplotlib"], check=False)

import torch
from revllm import Config, GPT, SavedTensorMeter
from data import prepare_tinystories, prepare_synthetic, TokenStream
import train as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SMOKE mode = tiny synthetic run that finishes in ~1 min on CPU. Auto-enabled when there is no GPU.
SMOKE = os.environ.get("SMOKE", "0") == "1" or DEVICE == "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "| SMOKE mode:", SMOKE)
os.makedirs("results", exist_ok=True)


In [ ]:
# @title Experiment configuration (shared by all notebooks)
if SMOKE:
    MODEL  = dict(vocab_size=512, block_size=64, n_layer=4, n_embd=128, n_head=4)
    TOKENS = 200_000          # token budget
    BATCH  = 16               # the fixed batch size for notebooks 01/02
    LR     = 2e-3
    LOG    = dict(eval_every=100, eval_iters=5, log_every=50)
else:
    # ~20.8M parameters (12.9M in the tied GPT-2 embedding, 7.9M in 10 transformer blocks of width 256)
    MODEL  = dict(vocab_size=50257, block_size=256, n_layer=10, n_embd=256, n_head=4)
    TOKENS = 50_000_000
    BATCH  = 32               # 32 x 256 = 8,192 tokens / step  -> ~6,100 steps for 50M tokens
    LR     = 6e-4
    LOG    = dict(eval_every=250, eval_iters=20, log_every=50)

def get_data():
    if SMOKE:
        return prepare_synthetic("data/synthetic", 2_000_000, 200_000, vocab=MODEL["vocab_size"])
    return prepare_tinystories("data/tinystories", n_train_tokens=55_000_000, n_val_tokens=2_000_000)

def gpu_table(rows, headers):
    w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |"
    print(line(headers)); print("|" + "|".join("-" * (x + 2) for x in w) + "|")
    for r in rows: print(line(r))

import matplotlib.pyplot as plt
def plot_runs(results, key="curve", title="training loss", smooth=25):
    plt.figure(figsize=(8, 4.5))
    for r in results:
        c = r[key]
        if not c: continue
        xs = [p[1] / 1e6 for p in c]; ys = [p[2] for p in c]
        if key == "curve" and smooth > 1 and len(ys) > smooth:
            ys = [sum(ys[max(0, i - smooth):i + 1]) / len(ys[max(0, i - smooth):i + 1]) for i in range(len(ys))]
        plt.plot(xs, ys, label=f"{r['run_name']}  (final {ys[-1]:.3f})")
    plt.xlabel("tokens seen (M)"); plt.ylabel("cross-entropy (nats)"); plt.title(title); plt.legend(); plt.grid(alpha=.3)
    plt.show()


In [ ]:
runs = []
for f in sorted(os.listdir("results")):
    if not f.endswith(".json"): continue
    r = json.load(open("results/" + f))
    if "curve" in r and not r["run_name"].endswith("_short"): runs.append(r)
lines = ["| run | mode | rev backprop | batch | steps | final train loss | final val loss | tokens/s | peak GB | minutes |", "|---|---|---|---|---|---|---|---|---|---|"]
for r in runs:
    c = r["config"]
    lines.append(f"| {r['run_name']} | {c['mode']} | {'yes' if c['mode']!='baseline' and c['rev_backprop'] else 'no'} | {r['batch_size']} | {r['steps']} | "
                 f"{r['final_train_loss']:.4f} | {r['final_val_loss']:.4f} | {r['tokens_per_s_steady']:,.0f} | "
                 f"{r['peak_mem_gb']:.2f} | {r['wall_time_s']/60:.1f} |" if r["peak_mem_gb"] else
                 f"| {r['run_name']} | {c['mode']} | {'yes' if c['mode']!='baseline' and c['rev_backprop'] else 'no'} | {r['batch_size']} | {r['steps']} | "
                 f"{r['final_train_loss']:.4f} | {r['final_val_loss']:.4f} | {r['tokens_per_s_steady']:,.0f} | n/a | {r['wall_time_s']/60:.1f} |")
hdr = [f"**GPU:** {runs[0]['gpu']}  **dtype:** {runs[0]['dtype']}  **params:** {runs[0]['n_params']/1e6:.2f}M  **tokens/run:** {runs[0]['tokens_seen']/1e6:.0f}M", ""] if runs else []
if os.path.exists("results/max_batch_probe.json"):
    p = json.load(open("results/max_batch_probe.json"))
    hdr += ["**Max batch probe** (%s, %.0f GB):" % (p["gpu"], p["total_gb"]), "", "| model | max batch | peak GB |", "|---|---|---|"]
    hdr += [f"| {k} | {v['max_batch']} | {v['peak_gb']:.2f} |" for k, v in p["probe"].items()] + [""]
md = "\n".join(hdr + lines)
open("results/summary.md", "w").write(md); print(md)
plot_runs(runs, title="training loss — all runs"); plt.savefig("results/loss_curves_all.png", dpi=120)
plot_runs(runs, key="val_curve", title="validation loss — all runs"); plt.savefig("results/val_curves_all.png", dpi=120)

In [ ]:
# On Colab: download the results folder so you can commit it to the repo
try:
    from google.colab import files
    import shutil; shutil.make_archive("results", "zip", "results"); files.download("results.zip")
except Exception as e:
    print("not on Colab (or download skipped):", e)